# Black Friday Sales Analysis

**Tabular Regression Project** — Predict purchase amounts from Black Friday sales data.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Black Friday (550,068 rows × 12 columns, sampled to 50,000)
Target: `Purchase`

## 2 · Project Overview

We predict the **Purchase amount** (in dollars) for Black Friday transactions using customer demographics and product category features. The dataset is large (550K rows) so we sample to 50,000 for manageable training times while maintaining representativeness.

This project emphasizes handling **categorical features**, **missing values** in product categories, and working with **large retail datasets**.

## 3 · Learning Objectives

1. Handle large datasets with row sampling.
2. Encode ordinal and nominal categorical features properly.
3. Handle missing values in product category columns.
4. Build and compare gradient-boosting models on retail data.
5. Use LazyPredict and FLAML for rapid model selection.

## 4 · Problem Statement

Given customer **demographics** (gender, age, city, marital status) and **product information** (category), predict the **Purchase amount** in a Black Friday sale. This helps retailers personalize offers and forecast revenue.

## 5 · Why This Project Matters

- **Retail analytics** is a core business application of ML.
- Understanding purchase drivers helps targeted marketing.
- Handling categorical features and missing values are essential skills.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 550,068 (sampled to 50,000) |
| **Columns** | 12 |
| **Features** | Gender, Age, Occupation, City_Category, Stay_In_Current_City_Years, Marital_Status, Product_Category_1/2/3 |
| **Target** | Purchase (continuous, USD) |
| **Missing** | Product_Category_2 (~31%), Product_Category_3 (~69%) |
| **File** | `train.csv` (local) |

## 7 · Dataset Source and License Notes

- **Source**: Analytics Vidhya / Kaggle Black Friday competition.
- **License**: Public for educational/competition use.
- **Note**: User_ID and Product_ID are identifiers — not useful as features.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'Purchase'
MAX_ROWS = 50000
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
DATA_PATH = os.path.join(SAVE_DIR, 'train.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
print(f'Original shape: {df.shape}')

if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f'Sampled to: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nMissing % Product_Category_2: {df["Product_Category_2"].isnull().mean()*100:.1f}%')
print(f'Missing % Product_Category_3: {df["Product_Category_3"].isnull().mean()*100:.1f}%')
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min()}, {df[TARGET].max()}]')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Gender')[TARGET].mean().plot(kind='bar', ax=axes[0,0], title='Avg Purchase by Gender')
df.groupby('Age')[TARGET].mean().plot(kind='bar', ax=axes[0,1], title='Avg Purchase by Age')
df.groupby('City_Category')[TARGET].mean().plot(kind='bar', ax=axes[1,0], title='Avg Purchase by City')
df.groupby('Occupation')[TARGET].mean().plot(kind='bar', ax=axes[1,1], title='Avg Purchase by Occupation')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title('Distribution of Purchase')

df.boxplot(column=TARGET, by='Gender', ax=axes[1])
axes[1].set_title('Purchase by Gender')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'target_analysis.png'), dpi=100)
plt.show()
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split Strategy

Simple 80/20 random split stratified by Gender to maintain demographic balance.

## 16 · Preprocessing Strategy

- Drop `User_ID` and `Product_ID` (identifiers).
- Encode `Gender`: M=1, F=0.
- Encode `Age`: ordinal mapping.
- Encode `City_Category`: label encode.
- Map `Stay_In_Current_City_Years`: '4+' → 4, then int.
- Fill missing `Product_Category_2/3` with 0 (absent category).

In [ ]:
# Drop IDs
df = df.drop(columns=['User_ID', 'Product_ID'])

# Encode Gender
df['Gender'] = df['Gender'].map({'M': 1, 'F': 0})

# Encode Age
age_map = {'0-17': 0, '18-25': 1, '26-35': 2, '36-45': 3, '46-50': 4, '51-55': 5, '55+': 6}
df['Age'] = df['Age'].map(age_map)

# Encode City_Category
df['City_Category'] = LabelEncoder().fit_transform(df['City_Category'])

# Stay years
df['Stay_In_Current_City_Years'] = df['Stay_In_Current_City_Years'].str.replace('+', '', regex=False).astype(int)

# Fill missing categories
df['Product_Category_2'] = df['Product_Category_2'].fillna(0).astype(int)
df['Product_Category_3'] = df['Product_Category_3'].fillna(0).astype(int)

print(f'Preprocessed shape: {df.shape}')
print(df.dtypes)

## 17 · Feature Engineering

We add interaction features between demographics and product categories.

In [ ]:
df['cat1_x_occupation'] = df['Product_Category_1'] * df['Occupation']
df['age_x_city'] = df['Age'] * df['City_Category']

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)

print(f'Baseline Linear Regression:')
print(f'  RMSE: {baseline_rmse:.2f}')
print(f'  MAE:  {baseline_mae:.2f}')
print(f'  R²:   {baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor

# Use a smaller sample for LazyPredict to avoid long runtimes
X_train_lp = X_train.head(5000)
y_train_lp = y_train.head(5000)
X_test_lp = X_test.head(1000)
y_test_lp = y_test.head(1000)

lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train_lp, X_test_lp, y_train_lp, y_test_lp)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.2, s=3)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=50, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.2, s=3)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **Product_Category_1** is the strongest predictor — certain product categories have much higher average purchase values.
- **Occupation** and **City_Category** also matter — urban professionals tend to spend more.
- Gender differences are modest after controlling for product category.
- Retailers can use these insights for targeted promotions and inventory planning.

## 26 · Limitations

- Purchase amounts are likely discretized or have specific pricing patterns.
- No temporal features — all transactions are from one Black Friday event.
- Product_ID was dropped — product-level modeling could improve results.
- R² on retail purchase data is typically modest due to inherent randomness.

## 27 · How to Improve This Project

1. Use Product_ID as a feature (target encoding).
2. Add user-level aggregation features (avg spend per user).
3. Try deeper hyperparameter tuning with Optuna.
4. Build a two-stage model: predict category first, then amount.
5. Use the full 550K rows with GPU-accelerated training.

## 28 · Production Considerations

- Real-time scoring for personalized pricing/offers.
- Need to refresh model for each sale event.
- Privacy: user demographics require careful handling.
- A/B testing framework to validate model-driven promotions.

## 29 · Common Mistakes

1. **Using User_ID/Product_ID as numeric features** — they are identifiers, not ordinal.
2. **Not handling '4+' in Stay_In_Current_City_Years** — string causes errors.
3. **Not filling missing Product_Category_2/3** — many models can't handle NaN.
4. **Using all 550K rows in LazyPredict** — too slow.

## 30 · Mini Challenge / Exercises

1. Try target encoding for Product_ID and see if R² improves.
2. Create age-bin × city interaction features.
3. Compare model performance on full 550K dataset vs. 50K sample.
4. Investigate which product categories have the largest prediction errors.

## 31 · Final Summary / Key Takeaways

- Gradient-boosting models handle mixed categorical/numerical retail data well.
- Feature engineering (interactions) provides modest improvements.
- Product category is the dominant signal for purchase amount.
- Sampling to 50K rows makes experimentation fast without major accuracy loss.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['baseline_linear_regression'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')